# Assignment 3 Q3: Inverting Embeddings (Colab)

Run cells top to bottom. The first run downloads CLIP-H and builds a cache of all one-word embeddings, so it can take a while. Reruns reuse the word cache.


In [ ]:
#@title Install dependencies and clone lab files
%pip -q install transformers accelerate

from pathlib import Path
import os

if not Path("llm_lab").exists():
    !git clone https://github.com/ethz-spylab/llm_lab.git
else:
    !git -C llm_lab pull
LAB_DIR = Path("llm_lab")


In [ ]:
#@title Runtime configuration
import torch
from pathlib import Path

BATCH_SIZE = 256  #@param {type:"integer"}
TOPN = 10         #@param {type:"integer"}
SAVE_TO_DRIVE = False  #@param {type:"boolean"}

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path('/content/drive/MyDrive/llm_assignment_3_q3')
else:
    OUTPUT_DIR = Path('.')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_dtype = torch.float16 if device.type == 'cuda' else torch.float32
WORD_CACHE = OUTPUT_DIR / '.cache' / 'q3_word_embeddings_laion_clip_h.npz'


In [ ]:
import json
import re
import time
from pathlib import Path

import numpy as np
import torch
from transformers import CLIPModel, CLIPTokenizer

MODEL_NAME = "laion/CLIP-ViT-H-14-laion2B-s32B-b79K"
KNOWN_EXAMPLE = "The solutions to this assignment"

SINGLE_WORD_THRESHOLD = 0.99
TWO_WORD_THRESHOLD = 0.99
MOVIE_THRESHOLD = 0.91
SCRAMBLED_THRESHOLD = 0.99


def threshold_for(idx):
    if idx < 4:
        return SINGLE_WORD_THRESHOLD
    if idx < 13:
        return TWO_WORD_THRESHOLD
    if idx < 19:
        return MOVIE_THRESHOLD
    return SCRAMBLED_THRESHOLD


def _load_default_movie_titles():
    """Load DEFAULT_MOVIE_TITLES from default_movies.txt.

    Tries paths next to the notebook first, then the current working directory.
    If the file is missing, returns an empty list and prints a warning — the
    list lives exclusively in default_movies.txt, not in the notebook source.
    """
    candidates = []
    try:
        candidates.append(LAB_DIR.parent / "default_movies.txt")
    except NameError:
        pass
    candidates += [
        Path("default_movies.txt"),
        Path("third/question3/default_movies.txt"),
        Path("question3/default_movies.txt"),
    ]
    for path in candidates:
        try:
            if path is None or not path.exists():
                continue
            titles = []
            for line in path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if line and not line.startswith("#"):
                    titles.append(line)
            if titles:
                print(f"Loaded {len(titles):,} default movie titles from {path}")
                return titles
        except Exception:
            continue
    print(
        f"WARNING: default_movies.txt not found in any of "
        f"{[str(p) for p in candidates if p]}. "
        f"DEFAULT_MOVIE_TITLES will be empty; set MOVIE_FILE or EXTRA_MOVIES "
        f"in the movies cell to supply candidates."
    )
    return []


DEFAULT_MOVIE_TITLES = _load_default_movie_titles()


def normalize_rows(x):
    x = x.astype(np.float32, copy=False)
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(norms, 1e-12)


def batched(items, batch_size):
    batch = []
    for item in items:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch


def embed_texts(texts, tokenizer, model, device):
    with torch.no_grad():
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
        inputs = {key: value.to(device) for key, value in inputs.items()}
        embeddings = model.get_text_features(**inputs)
        if not isinstance(embeddings, torch.Tensor):
            # Match safety_checker.get_embedding exactly: use pooler_output WITHOUT
            # applying text_projection. The staff's bad_embeds were generated
            # via this exact branch when get_text_features returned a non-Tensor.
            embeddings = embeddings.pooler_output
        embeddings = embeddings / embeddings.norm(p=2, dim=-1, keepdim=True)
        return embeddings.detach().cpu().float().numpy()


def embed_many(texts, tokenizer, model, device, batch_size, label):
    chunks = []
    start = time.time()
    for i, batch in enumerate(batched(texts, batch_size), start=1):
        chunks.append(embed_texts(batch, tokenizer, model, device))
        done = min(i * batch_size, len(texts))
        if i == 1 or done == len(texts) or done % (batch_size * 20) == 0:
            print(f"{label}: embedded {done:,}/{len(texts):,} texts in {time.time() - start:.1f}s")
    return np.concatenate(chunks, axis=0)


def topk_indices(scores, k):
    k = min(k, scores.shape[0])
    unordered = np.argpartition(scores, -k)[-k:]
    return unordered[np.argsort(scores[unordered])[::-1]]


def top_records(texts, scores, k):
    return [{"text": texts[i], "score": float(scores[i])} for i in topk_indices(scores, k)]


def cache_key(model_name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", model_name).strip("_")


def load_or_build_word_embeddings(words, cache_path, tokenizer, model, device, batch_size):
    cache_path = Path(cache_path)
    if cache_path.exists():
        data = np.load(cache_path, allow_pickle=False)
        cached_words = data["words"].astype(str).tolist()
        if cached_words == words:
            print(f"Loaded word embeddings from {cache_path}")
            return data["embeddings"].astype(np.float32, copy=False)
        print(f"Ignoring stale word cache at {cache_path}; word list changed.")
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    embeddings = embed_many(words, tokenizer, model, device, batch_size, "word cache")
    np.savez(cache_path, words=np.array(words), embeddings=embeddings.astype(np.float32))
    print(f"Saved word embeddings to {cache_path}")
    return embeddings


def shortlist_pairs(words, embeddings, target, shortlist_size, sign_invariant=False):
    sim = embeddings @ embeddings.T
    denom = np.sqrt(np.maximum(2.0 + 2.0 * sim, 1e-12))
    if sign_invariant:
        word_target_scores = np.abs(embeddings) @ np.abs(target)
    else:
        word_target_scores = embeddings @ target
    approx_scores = (word_target_scores[:, None] + word_target_scores[None, :]) / denom
    flat_indices = topk_indices(approx_scores.ravel(), shortlist_size)
    row_indices, col_indices = np.unravel_index(flat_indices, approx_scores.shape)
    seen = set()
    phrases = []
    for row, col in zip(row_indices, col_indices):
        for first, second in ((words[row], words[col]), (words[col], words[row])):
            phrase = f"{first} {second}"
            if phrase not in seen:
                seen.add(phrase)
                phrases.append(phrase)
    return phrases



def _morph(word, words_set):
    """Return word + common morphological/derivational variants in words_set."""
    out = {word}
    w = word
    # Inflectional: plurals
    if w.endswith("s") and len(w) > 2 and w[:-1] in words_set: out.add(w[:-1])
    if w + "s" in words_set: out.add(w + "s")
    if w.endswith("es") and len(w) > 3 and w[:-2] in words_set: out.add(w[:-2])
    if w.endswith("ies") and w[:-3] + "y" in words_set: out.add(w[:-3] + "y")
    if w.endswith("y") and w[:-1] + "ies" in words_set: out.add(w[:-1] + "ies")
    # Inflectional: verb forms
    if w.endswith("ed") and len(w) > 3 and w[:-2] in words_set: out.add(w[:-2])
    if w.endswith("ed") and len(w) >= 4 and w[:-1] in words_set: out.add(w[:-1])
    if w.endswith("ing") and len(w) > 4 and w[:-3] in words_set: out.add(w[:-3])
    if w.endswith("ing") and len(w) > 4 and w[:-3] + "e" in words_set: out.add(w[:-3] + "e")
    # Derivational stripping (noisier but only adds candidates)
    for suf in ("al", "age", "ic", "ous", "ly", "ness", "ment", "ation",
                "tion", "er", "or", "ish", "ful"):
        if w.endswith(suf) and len(w) > len(suf) + 1 and w[:-len(suf)] in words_set:
            out.add(w[:-len(suf)])
    # Derivational addition
    for suf in ("al", "age", "y", "ic"):
        if w + suf in words_set: out.add(w + suf)
    return out


def _compound_splits(word, words_set):
    """Yield (left, right) where both halves are in words_set and >=2 chars."""
    for i in range(2, len(word) - 1):
        left, right = word[:i], word[i:]
        if left in words_set and right in words_set:
            yield left, right


def _expand_side(word, words_set):
    """All single-word variants of `word`: morphology, splits, and morph of splits."""
    out = set(_morph(word, words_set))
    for left, right in _compound_splits(word, words_set):
        out.add(left); out.add(right)
        out |= _morph(left, words_set)
        out |= _morph(right, words_set)
    return out


def refine_two_word_candidate(phrase, words_set):
    """Expand a 2-word phrase via morphology + compound-split of each word.

    Handles cases where the symmetric search returned the wrong morphological
    variant: 'postal offices' -> 'post office', 'lunch boxes' -> 'lunch box',
    'icecream icecream' -> 'ice cream'.
    """
    parts = phrase.split()
    if len(parts) != 2:
        return [phrase]
    a_var = _expand_side(parts[0], words_set)
    b_var = _expand_side(parts[1], words_set)
    return sorted({f"{a} {b}" for a in a_var for b in b_var})


def score_text_candidates(texts, score_fn, tokenizer, model, device, batch_size, topn, label):
    best_texts = []
    best_scores = np.array([], dtype=np.float32)
    start = time.time()
    done = 0
    for batch in batched(texts, batch_size):
        scores = score_fn(embed_texts(batch, tokenizer, model, device))
        if best_scores.size == 0:
            combined_texts = batch
            combined_scores = scores
        else:
            combined_texts = best_texts + batch
            combined_scores = np.concatenate([best_scores, scores])
        keep = topk_indices(combined_scores, topn)
        best_texts = [combined_texts[i] for i in keep]
        best_scores = combined_scores[keep]
        done += len(batch)
        if done == len(batch) or done == len(texts) or done % max(batch_size * 50, 1) == 0:
            print(f"{label}: scored {done:,}/{len(texts):,}; best={best_texts[0]!r} score={best_scores[0]:.6f}; {time.time() - start:.1f}s")
    return [{"text": text, "score": float(score)} for text, score in zip(best_texts, best_scores)]


def add_report_entries(report, guesses, entries_by_index):
    for idx, entries in entries_by_index.items():
        report[str(idx)] = entries
        if entries:
            guesses[idx] = entries[0]["text"]
            score = entries[0]["score"]
            thr = threshold_for(idx)
            tag = "OK" if score >= thr else "BELOW"
            note = " (heuristic; target is masked)" if idx == 19 else ""
            print(f"[{idx:02d}] selected {guesses[idx]!r} score={score:.6f} thr={thr:.2f} [{tag}]{note}")


In [ ]:
bad_embeddings = normalize_rows(np.load(LAB_DIR / "data" / "bad_embeddings.npy"))
words = [line.strip() for line in (LAB_DIR / "data" / "corncob_lowercase.txt").read_text().splitlines() if line.strip()]

print("bad_embeddings", bad_embeddings.shape)
print("word list", len(words))
print("device", device)

tokenizer = CLIPTokenizer.from_pretrained(MODEL_NAME)
model = CLIPModel.from_pretrained(MODEL_NAME, torch_dtype=model_dtype)
model.eval().to(device)

# Sanity check: bad_embeddings[20] is the staff's encoding of KNOWN_EXAMPLE. If
# our tokenizer/model/normalization pipeline matches theirs, cos sim should be ~1.
_known_embed = embed_texts([KNOWN_EXAMPLE], tokenizer, model, device)[0]
_known_sim = float(_known_embed @ bad_embeddings[20])
print(f"KNOWN_EXAMPLE cos sim with bad_embeddings[20]: {_known_sim:.6f}")
assert _known_sim > 0.99, (
    f"Encoding pipeline mismatch (cos sim {_known_sim:.4f}); "
    "check tokenizer/model/dtype/normalization."
)

WORD_CACHE.parent.mkdir(parents=True, exist_ok=True)
word_embeddings = load_or_build_word_embeddings(words, WORD_CACHE, tokenizer, model, device, BATCH_SIZE)
word_embeddings = normalize_rows(word_embeddings)

report = {}
guesses = ["" for _ in range(21)]
guesses[20] = KNOWN_EXAMPLE


## 1. Single-word concepts: `bad_embeddings[0:4]`


In [ ]:
entries_by_index = {}
for idx in range(4):
    scores = word_embeddings @ bad_embeddings[idx]
    entries_by_index[idx] = top_records(words, scores, TOPN)
add_report_entries(report, guesses, entries_by_index)


## 2. Two-word concepts: `bad_embeddings[4:13]`


In [ ]:
# Two-pass two-word search:
#   1) Symmetric escalating (existing) — generate candidates from the top-N
#      seed pool and exact-CLIP-score them.
#   2) Refinement — for the best phrase so far, expand each word through
#      morphology + compound splits and re-score the variants. Catches the
#      'postal offices' -> 'post office' / 'lunch boxes' -> 'lunch box' /
#      'icecream icecream' -> 'ice cream' failure modes.
PAIR_ESCALATION_STEPS = [(2500, 10000), (5000, 25000), (8000, 50000)]
REFINE_TOP_K = 5  # how many top phrases to expand variants from

words_set = set(words)

entries_by_index = {}
for idx in range(4, 13):
    target = bad_embeddings[idx]
    score_fn = lambda embeddings, target=target: embeddingsings @ target
    best_entries = []

    # PASS 1: symmetric escalating
    for pool_size, shortlist_size in PAIR_ESCALATION_STEPS:
        seed_scores = word_embeddings @ target
        pool_indices = topk_indices(seed_scores, pool_size)
        pool_words = [words[i] for i in pool_indices]
        pool_embeddings = word_embeddings[pool_indices]
        candidates = shortlist_pairs(pool_words, pool_embeddings, target, shortlist_size)
        print(f"[{idx:02d}] sym pool={pool_size} shortlisted {len(candidates):,} phrases")
        entries = score_text_candidates(
            candidates, score_fn, tokenizer, model, device, BATCH_SIZE, TOPN,
            f"[{idx:02d}] sym p{pool_size}",
        )
        if entries and (not best_entries or entries[0]["score"] > best_entries[0]["score"]):
            best_entries = entries
        if best_entries and best_entries[0]["score"] >= TWO_WORD_THRESHOLD:
            break
        print(f"[{idx:02d}] sym best so far = {best_entries[0]['score']:.6f}; escalating")

    # PASS 2: refinement (morphology + compound splits of top-K phrases)
    if best_entries and best_entries[0]["score"] < TWO_WORD_THRESHOLD:
        refine_seeds = [e["text"] for e in best_entries[:REFINE_TOP_K]]
        seen = set()
        refine_candidates = []
        for seed_phrase in refine_seeds:
            for variant in refine_two_word_candidate(seed_phrase, words_set):
                if variant not in seen:
                    seen.add(variant)
                    refine_candidates.append(variant)
        print(f"[{idx:02d}] refine: {len(refine_candidates):,} morph/split variants of top {len(refine_seeds)}")
        entries = score_text_candidates(
            refine_candidates, score_fn, tokenizer, model, device, BATCH_SIZE, TOPN,
            f"[{idx:02d}] refine",
        )
        if entries and entries[0]["score"] > best_entries[0]["score"]:
            best_entries = entries

    entries_by_index[idx] = best_entries
add_report_entries(report, guesses, entries_by_index)


## 3. Movie-title concepts: `bad_embeddings[13:19]`

    The built-in list is intentionally small. For better coverage, upload or point `MOVIE_FILE` to a larger one-title-per-line file and rerun this cell.


In [ ]:
MOVIE_FILE = ""  # optional: path to a text file with one movie title per line
EXTRA_MOVIES = []  # optional: add strings here, e.g. ["The Great Dictator", "Heat"]

seen = set()
movie_candidates = []

def add_movie(title):
    title = title.strip()
    if not title or title.startswith("#"):
        return
    key = title.casefold()
    if key not in seen:
        seen.add(key)
        movie_candidates.append(title)

for title in DEFAULT_MOVIE_TITLES:
    add_movie(title)
if MOVIE_FILE:
    for line in Path(MOVIE_FILE).read_text(encoding="utf-8", errors="ignore").splitlines():
        add_movie(line)
for title in EXTRA_MOVIES:
    add_movie(title)

print(f"movie candidates: {len(movie_candidates):,}")
movie_embeddings = embed_many(movie_candidates, tokenizer, model, device, BATCH_SIZE, "movie candidates")
entries_by_index = {}
for idx in range(13, 19):
    scores = movie_embeddings @ bad_embeddings[idx]
    entries_by_index[idx] = top_records(movie_candidates, scores, TOPN)
add_report_entries(report, guesses, entries_by_index)


## 4. Sign-masked two-word concept: `bad_embeddings[19]`


In [ ]:
SCRAMBLED_POOL_SIZE = 4000
SCRAMBLED_SHORTLIST_SIZE = 30000
SCRAMBLED_REFINE_TOP_K = 10

# Re-create words_set if the two-word cell hasn't been run in this kernel session.
try:
    words_set
except NameError:
    words_set = set(words)

idx = 19
target = bad_embeddings[idx]
target_abs = np.abs(target)
seed_scores = np.abs(word_embeddings) @ target_abs
pool_indices = topk_indices(seed_scores, SCRAMBLED_POOL_SIZE)
pool_words = [words[i] for i in pool_indices]
pool_embeddings = word_embeddings[pool_indices]
candidates = shortlist_pairs(
    pool_words, pool_embeddings, target,
    SCRAMBLED_SHORTLIST_SIZE, sign_invariant=True,
)
print(f"[{idx:02d}] exact-scoring {len(candidates):,} sign-invariant phrases from {len(pool_words):,} seed words")


# Two complementary sign-invariant heuristics. Both are invariant to the random
# +/-1 mask but have different false-positive structure, so cross-checking the
# top-K of each gives a stronger signal than either alone.
def score_dot_abs(embeddings):
    return np.abs(embeddings) @ target_abs


def score_neg_l2_abs(embeddings):
    return -np.linalg.norm(np.abs(embeddings) - target_abs[None, :], axis=1)


# PASS 1: bulk search
entries_dot = score_text_candidates(
    candidates, score_dot_abs, tokenizer, model, device, BATCH_SIZE, TOPN,
    f"[{idx:02d}] sym <|c|,|t|>",
)
entries_l2 = score_text_candidates(
    candidates, score_neg_l2_abs, tokenizer, model, device, BATCH_SIZE, TOPN,
    f"[{idx:02d}] sym -||(|c|-|t|)||",
)

print("PASS 1 top by <|c|,|t|>:")
for e in entries_dot[:5]:
    print(f"  {e['score']:.6f}  {e['text']!r}")
print("PASS 1 top by -||(|c|-|t|)||:")
for e in entries_l2[:5]:
    print(f"  {e['score']:.6f}  {e['text']!r}")


# PASS 2: refine the top-K candidates from each ranking via morphology +
# compound splits. Mirrors the two-word refinement pass — catches plural/
# singular and split-compound variants the bulk search missed.
if entries_dot and entries_dot[0]["score"] < SCRAMBLED_THRESHOLD:
    refine_seeds = []
    seen = set()
    for src_list in (entries_dot[:SCRAMBLED_REFINE_TOP_K],
                     entries_l2[:SCRAMBLED_REFINE_TOP_K]):
        for e in src_list:
            if e["text"] not in seen:
                seen.add(e["text"])
                refine_seeds.append(e["text"])

    refine_candidates = []
    seen = set()
    for seed_phrase in refine_seeds:
        for variant in refine_two_word_candidate(seed_phrase, words_set):
            if variant not in seen:
                seen.add(variant)
                refine_candidates.append(variant)

    print(f"[{idx:02d}] refine: {len(refine_candidates):,} morph/split variants of top {len(refine_seeds)}")
    entries_dot_refined = score_text_candidates(
        refine_candidates, score_dot_abs, tokenizer, model, device, BATCH_SIZE, TOPN,
        f"[{idx:02d}] refine <|c|,|t|>",
    )
    entries_l2_refined = score_text_candidates(
        refine_candidates, score_neg_l2_abs, tokenizer, model, device, BATCH_SIZE, TOPN,
        f"[{idx:02d}] refine -||(|c|-|t|)||",
    )

    if entries_dot_refined and entries_dot_refined[0]["score"] > entries_dot[0]["score"]:
        entries_dot = entries_dot_refined
    if entries_l2_refined and entries_l2_refined[0]["score"] > entries_l2[0]["score"]:
        entries_l2 = entries_l2_refined

    print("PASS 2 (after refinement) top by <|c|,|t|>:")
    for e in entries_dot[:5]:
        print(f"  {e['score']:.6f}  {e['text']!r}")
    print("PASS 2 (after refinement) top by -||(|c|-|t|)||:")
    for e in entries_l2[:5]:
        print(f"  {e['score']:.6f}  {e['text']!r}")

# Default guess uses the dot-abs ranking (preserves prior behaviour); the
# L2-distance ranking is stored alongside so you can swap manually if the
# top candidates differ.
entries_by_index = {idx: entries_dot}
report["19_neg_l2_abs_topk"] = entries_l2
add_report_entries(report, guesses, entries_by_index)


## 5. Save submission file


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

missing = [i for i in range(20) if not guesses[i]]
if missing:
    print(f"WARNING: missing guesses for indices {missing}; filling with placeholders that will not pass grading.")
    for i in missing:
        guesses[i] = f"TODO missing guess {i}"
guesses[20] = KNOWN_EXAMPLE

(OUTPUT_DIR / "Q3_guesses.txt").write_text("\n".join(guesses) + "\n", encoding="utf-8")
(OUTPUT_DIR / "Q3_search_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

print("Wrote", OUTPUT_DIR / "Q3_guesses.txt")
print("Wrote", OUTPUT_DIR / "Q3_search_report.json")

print()
print("Per-index summary:")
ok_count = 0
for idx in range(21):
    entries = report.get(str(idx)) or []
    score = entries[0]["score"] if entries else float("nan")
    thr = threshold_for(idx)
    if idx == 20:
        status = "GIVEN"
    elif not entries:
        status = "MISSING"
    elif score >= thr:
        status = "OK"
        ok_count += 1
    else:
        status = "BELOW"
    note = "  *|c|,|t| heuristic*" if idx == 19 else ""
    print(f"  [{idx:02d}] {status:7s} score={score:.4f} thr={thr:.2f}  {guesses[idx]!r}{note}")
print()
print(f"Passed local threshold check: {ok_count}/20 (idx 20 is the given example).")
print("Note: idx 19's score is the sign-invariant heuristic, not the true cos sim against the unmasked target.")
